In [6]:
# # Optimization of Shallow Neural Networks for Image Recognition

import tensorflow as tf
from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
import time
from sklearn.metrics import classification_report, accuracy_score
import optuna
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

# ## Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Flatten the images for shallow networks
x_train_flat = x_train.reshape(-1, 28 * 28)
x_test_flat = x_test.reshape(-1, 28 * 28)

# ## Function to count parameters
def count_params(model):
    return model.count_params()

# ## Function to compute FLOPs (Updated to work with TF 2.x)
def compute_flops(model):
    """
    Compute FLOPs for a given TensorFlow model.
    """
    # Convert the model to a concrete function
    concrete_func = tf.function(model).get_concrete_function(tf.TensorSpec([1] + list(model.input_shape[1:]), tf.float32))

    # Freeze the model (convert variables to constants)
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    frozen_graph = frozen_func.graph

    # Use TensorFlow profiler to compute FLOPs
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

    # Compute FLOPs from the frozen graph
    flops = tf.compat.v1.profiler.profile(graph=frozen_graph, run_meta=run_meta, options=opts)

    return flops.total_float_ops if flops else 0


# ## Define Shallow Neural Network Model
def create_shallow_model(input_size, hidden_units, activation, dropout_rate, output_size):
    model = models.Sequential()
    model.add(layers.InputLayer(shape=(input_size,)))  # Updated input layer definition
    for units in hidden_units:
        model.add(layers.Dense(units, activation=activation))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(output_size, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ## Optuna Objective Function for Bayesian Optimization
def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_units = [trial.suggest_int(f"n_units_l{i}", 32, 128, step=32) for i in range(n_layers)]
    activation = trial.suggest_categorical("activation", ["relu", "sigmoid", "tanh"])
    dropout_rate = trial.suggest_float("dropout", 0.0, 0.5)

    model = create_shallow_model(input_size=784, hidden_units=hidden_units, activation=activation,
                                 dropout_rate=dropout_rate, output_size=10)
    
    start_time = time.time()
    history = model.fit(x_train_flat, y_train, epochs=5, validation_data=(x_test_flat, y_test), verbose=0)
    end_time = time.time()

    training_time = end_time - start_time
    y_pred = model.predict(x_test_flat)
    y_pred_classes = np.argmax(y_pred, axis=1)

    classification_metrics = classification_report(y_test, y_pred_classes, output_dict=True)
    precision = classification_metrics['weighted avg']['precision']
    recall = classification_metrics['weighted avg']['recall']
    f1_score = classification_metrics['weighted avg']['f1-score']
    final_accuracy = accuracy_score(y_test, y_pred_classes)
    final_loss = history.history['val_loss'][-1]

    num_params = count_params(model)
    num_flops = compute_flops(model)

    trial.set_user_attr("hidden_units", hidden_units)
    trial.set_user_attr("training_time", training_time)
    trial.set_user_attr("accuracy", final_accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1_score", f1_score)
    trial.set_user_attr("final_loss", final_loss)
    trial.set_user_attr("num_params", num_params)
    trial.set_user_attr("num_flops", num_flops)

    return final_accuracy

# ## Run Bayesian Optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Collect results
optuna_results = []
for trial in study.trials:
    optuna_results.append({
        'Configuration': f"Optuna - {trial.number}",
        'Accuracy': trial.user_attrs['accuracy'],
        'Loss': trial.user_attrs['final_loss'],
        'Precision': trial.user_attrs['precision'],
        'Recall': trial.user_attrs['recall'],
        'F1-Score': trial.user_attrs['f1_score'],
        'Training Time (s)': trial.user_attrs['training_time'],
        'Hidden Units': trial.user_attrs['hidden_units'],
        'Num Params': trial.user_attrs['num_params'],
        'Num FLOPs': trial.user_attrs['num_flops']
    })

df_optuna = pd.DataFrame(optuna_results)
df_optuna.to_excel('optimisation_results_with_metrics.xlsx', index=False)

# ## Train CNN Baseline for Comparison
def create_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = create_cnn()
cnn_model.fit(x_train.reshape(-1, 28, 28, 1), y_train, epochs=5, validation_data=(x_test.reshape(-1, 28, 28, 1), y_test), verbose=0)

# ## Save CNN Performance Metrics
cnn_params = count_params(cnn_model)
cnn_flops = compute_flops(cnn_model)
cnn_acc = cnn_model.evaluate(x_test.reshape(-1, 28, 28, 1), y_test, verbose=0)[1]

df_cnn = pd.DataFrame([{
    'Configuration': 'CNN Benchmark',
    'Accuracy': cnn_acc,
    'Num Params': cnn_params,
    'Num FLOPs': cnn_flops
}])

df_combined = pd.concat([df_optuna, df_cnn], ignore_index=True)
df_combined.to_excel('optimisation_results_with_metrics.xlsx', index=False)

print("Optimization and CNN benchmark results saved successfully.")


[I 2025-03-18 15:52:41,455] A new study created in memory with name: no-name-d7b8b768-c3a8-4542-8484-a1662f1e0b1e


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 677us/step
Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


[I 2025-03-18 15:52:53,980] Trial 0 finished with value: 0.949 and parameters: {'n_layers': 3, 'n_units_l0': 128, 'n_units_l1': 96, 'n_units_l2': 32, 'activation': 'sigmoid', 'dropout': 0.49613209139933795}. Best is trial 0 with value: 0.949.


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 596us/step


[I 2025-03-18 15:53:04,951] Trial 1 finished with value: 0.9682 and parameters: {'n_layers': 3, 'n_units_l0': 64, 'n_units_l1': 32, 'n_units_l2': 64, 'activation': 'tanh', 'dropout': 0.09596007446532973}. Best is trial 1 with value: 0.9682.


...

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 615us/step


[I 2025-03-18 15:56:09,619] Trial 18 finished with value: 0.9744 and parameters: {'n_layers': 1, 'n_units_l0': 96, 'activation': 'relu', 'dropout': 0.28788268754905083}. Best is trial 12 with value: 0.9782.


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 557us/step


[I 2025-03-18 15:56:19,147] Trial 19 finished with value: 0.9732 and parameters: {'n_layers': 1, 'n_units_l0': 64, 'activation': 'relu', 'dropout': 0.1205590584011651}. Best is trial 12 with value: 0.9782.
C:\Users\Mihir\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Optimization and CNN benchmark results saved successfully.
